# ETL Pipeline in Colab

This notebook demonstrates a Python ETL pipeline using pandas and scikit-learn. It is designed to run smoothly in Google Colab.


In [ ]:
# Install required packages for Colab if not already installed
%pip install -q pandas scikit-learn


In [ ]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline


## Mount Google Drive and Load Sample Data
This step attempts to mount Google Drive and load a CSV from `MyDrive`. If the file is not found, a synthetic DataFrame is created for demonstration.


In [ ]:
drive_root = Path('/content/drive')
data_path = Path('/content/drive/MyDrive/data/sales_data.csv')

if not drive_root.exists():
    print('Google Drive is not mounted yet. The notebook will create sample data locally.')
else:
    print('Google Drive is available. Checking for sample file.')

def create_sample_data():
    return pd.DataFrame({
        'order_id': [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010],
        'customer_id': [501, 502, 501, 503, 504, 502, 505, 506, 504, 507],
        'product': ['Widget', 'Gadget', 'Widget', 'Widget', 'Doohickey', 'Gadget', 'Widget', None, 'Doohickey', 'Gadget'],
        'category': ['A', 'B', 'A', 'A', 'C', 'B', 'A', 'C', None, 'B'],
        'quantity': [10, 5, 12, 7, 3, np.nan, 10, 8, 4, 6],
        'unit_price': [9.99, 19.99, 9.99, 9.99, 14.99, 19.99, np.nan, 14.99, 14.99, 19.99],
        'order_date': ['2024-01-12', '2024-01-15', '2024-02-02', '2024-02-05', '2024-02-10', '2024-03-01', '2024-03-08', '2024-03-12', '2024-03-15', '2024-04-01'],
        'region': ['North', 'West', 'North', 'East', 'South', 'West', 'East', 'South', 'South', 'West']
    })

if data_path.exists():
    df = pd.read_csv(data_path)
    print('Loaded data from Drive:', data_path)
else:
    print('Data file not found in Drive; using generated sample data.')
    df = create_sample_data()

df.head()


## Inspect and Clean Data
Inspect the structure of the DataFrame, handle missing values, drop duplicates, and correct data types.


In [ ]:
print('Shape:', df.shape)
print('\nData types:')
print(df.dtypes)
print('\nMissing values:')
print(df.isna().sum())
print('\nSample rows:')
display(df.head())

df = df.drop_duplicates()
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
df['product'] = df['product'].fillna('Unknown')
df['category'] = df['category'].fillna('Unknown')
df['quantity'] = df['quantity'].astype(float).fillna(df['quantity'].median())
df['unit_price'] = df['unit_price'].astype(float).fillna(df['unit_price'].median())
df['region'] = df['region'].fillna('Unknown')
print('\nCleaned missing values:')
print(df.isna().sum())
display(df.head())


## Transform Data with Pandas
Create new features, filter rows, and aggregate data for analysis or downstream loading.


In [ ]:
df['revenue'] = df['quantity'] * df['unit_price']
df['order_year'] = df['order_date'].dt.year
df['order_month'] = df['order_date'].dt.month
df['weekday'] = df['order_date'].dt.day_name()

df = df[df['revenue'] > 0].copy()

sales_by_product = df.groupby('product', as_index=False).agg(
    total_quantity=('quantity', 'sum'),
    total_revenue=('revenue', 'sum'),
    order_count=('order_id', 'count')
)
sales_by_region = df.groupby('region', as_index=False).agg(
    total_revenue=('revenue', 'sum'),
    average_order_value=('revenue', 'mean')
)
print('Sales by product:')
display(sales_by_product)
print('\nSales by region:')
display(sales_by_region)

pivot_table = df.pivot_table(
    index='region',
    columns='product',
    values='revenue',
    aggfunc='sum',
    fill_value=0
).reset_index()
print('\nRevenue pivot table:')
display(pivot_table)


## Export Processed Data to Drive
Save the cleaned and transformed DataFrame back to Google Drive or locally if Drive is unavailable.


In [ ]:
output_folder = Path('/content/drive/MyDrive/data') if drive_root.exists() else Path('.')
output_folder.mkdir(parents=True, exist_ok=True)
cleaned_path = output_folder / 'processed_sales.csv'
pivot_path = output_folder / 'sales_pivot.csv'

df.to_csv(cleaned_path, index=False)
pivot_table.to_csv(pivot_path, index=False)
print(f'Saved cleaned dataset to: {cleaned_path}')
print(f'Saved pivot dataset to:   {pivot_path}')
